In [20]:
import numpy as np
import matplotlib.pyplot as plt
import Basic_functions as bf
from scipy.sparse import vstack


import torch as pt
import optuna as opt
import sklearn.metrics as skm
import sklearn.utils as sku
from sklearn.preprocessing import StandardScaler

In [91]:
import torch.nn as nn
from torch.nn import CrossEntropyLoss
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchao.sparsity.training import (
    SemiSparseLinear,
    SemiSparseActivationLinear,
    swap_linear_with_semi_sparse_linear,
    swap_semi_sparse_linear_with_linear,
)
#from torch.nn import L1Loss
#from scipy.special import expit

In [ ]:
trigrams = np.genfromtxt('txt_files/trigons_filtered.txt', delimiter='\n', dtype=str)
data_speeches, labels_speeches = bf.read_files('Speeches', trigrams, sparse=True)
data_articles, labels_articles = bf.read_files('articles', trigrams, sparse=True)

In [58]:
data_comb = vstack([data_speeches, data_articles], format='csr')
labels_comb = np.concatenate([labels_speeches, labels_articles])
labels_SvsA = np.concatenate([np.ones(len(labels_speeches)), np.zeros(len(labels_articles))])

In [59]:
data_comb, labels_comb = sku.shuffle(data_comb, labels_SvsA, random_state=42)

In [60]:
data_train, data_train_labels, data_val, data_val_labels, data_test, data_test_labels = train_val_test_split(data_comb, labels_comb, val_size=0.20, test_size=0.05)

In [73]:
data_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 65924800 stored elements and shape (141773, 5606)>

In [93]:
class MyDataset(Dataset):    
    def __init__(self, X_data, y_data):
        self.input = X_data
        self.truth = y_data
        
    def __getitem__(self, index):
        return self.input[index], self.truth[index]
        
    def __len__ (self):
        return self.truth.shape[0]

#In pytorch, there is an additional step of turning your data into tensors
train_data = MyDataset(data_train, data_train_labels)
val_data = MyDataset(data_val, data_val_labels)

In [99]:
# Define the model:
class TestModel(nn.Module):
    def __init__(self):
        super(TestModel, self).__init__()        # Here we define the layers.
        self.input_layer = nn.Linear(len(trigrams), 192)     #In pytorch, you define the input and output edges.
        self.hidden_layer1 = nn.Linear(192, 48)
        self.hidden_layer2 = nn.Linear(48, 12)
        self.output_layer = nn.Linear(12, 2)
        self.relu = nn.ReLU()
        
    def forward(self, inputs):                  # Here we define how data passes through the layers. 
        x = self.input_layer(inputs)            # Also here, pytorch is a bit more explicit in defining the layers and activation function separately
        x = self.relu(x)
        x = self.hidden_layer1(x)
        x = self.relu(x)
        x = self.hidden_layer2(x)
        x = self.relu(x)
        x = self.output_layer(x)
        return x

In [100]:
# Training loop:
def Train(model, optimizer, loss_function, train_loader, validation_loader, device, epochs):
    validation_loss = []
    training_loss   = []
    model.train()
    for e in range(0, epochs):
        epoch_loss = 0
        n_minibatches = 0
        for input_train_batch, truth_train_batch in train_loader:
            input_train_batch, truth_train_batch = input_train_batch.to(device), truth_train_batch.to(device)
            optimizer.zero_grad()
            prediction = model(input_train_batch)  # This asks our model to produce predictions on the training batch            
            loss = loss_function(prediction, truth_train_batch.long())  # This calculates the loss
            loss.backward()                                             # This initiates the backpropagation
            optimizer.step()
            epoch_loss += loss.item()
            n_minibatches += 1
        
        # Now that the model have trained 1 epoch, we evaluate the model on the validation set:
        valid_loss = Validate(model, validation_loader, device, loss_function)
        validation_loss.append(valid_loss)
        training_loss.append(epoch_loss/n_minibatches)
        print('EPOCH: %s | training loss: %s  | validation loss: %s'%(e+1,round(epoch_loss/n_minibatches,3), round(valid_loss, 3)))
    return training_loss, validation_loss


def Validate(model, validation_loader, device, loss_function):
    model.eval()
    n_batches  = 0
    validation_loss = 0
    with pt.no_grad():
        for input_valid_batch, truth_valid_batch in validation_loader:
            input_valid_batch, truth_valid_batch = input_valid_batch.to(device), truth_valid_batch.to(device)
            prediction = model(input_valid_batch)
            loss = loss_function(prediction, truth_valid_batch.long())
            validation_loss += loss.item()
            n_batches += 1
    validation_loss = validation_loss/n_batches
    return validation_loss


def Predict(model, prediction_loader, device):
    model.eval()
    predictions = []
    print('PREDICTING!')
    with pt.no_grad():
        for input_pred_batch, _ in prediction_loader:
            input_pred_batch = input_pred_batch.to(device)
            prediction = model(input_pred_batch)
            predictions.extend(prediction.numpy())
    print('Done Predicting!')
    return predictions
                
class MeanRelativeAbsoluteDeviationLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
                
    def forward(self, prediction, target):
        prediction = prediction.view_as(target)
        relative_absolute_error = pt.abs(prediction - target) / (pt.abs(target) + self.eps)
        return pt.mean(relative_absolute_error)

In [101]:
def sparse_collate(batch):
    xs, ys = zip(*batch)

    first = xs[0]
    n_features = first.shape[-1]

    batch_rows = []
    batch_cols = []
    batch_vals = []

    for i, x in enumerate(xs):
        x = x.tocoo()
        batch_rows.append(np.full_like(x.col, i, dtype=np.int64))
        batch_cols.append(x.col.astype(np.int64))
        batch_vals.append(x.data.astype(np.float32))

    indices = pt.tensor(
        np.vstack([np.concatenate(batch_rows), np.concatenate(batch_cols)]),
        dtype=pt.long
    )
    values = pt.tensor(np.concatenate(batch_vals), dtype=pt.float32)

    X = pt.sparse_coo_tensor(indices, values, size=(len(xs), n_features)).coalesce()
    y = pt.tensor(np.asarray(ys), dtype=pt.long)
    return X, y

In [102]:
# Now everything is ready, and we thus define the optimisation (hyper-) parameters:
learning_rate = 1e-3      # The step size in the direction of "good" from stocastic gradient descent (important!)
batch_size    = 32        # The size of the batches used for each of the stocastic gradient descent calculations
n_epochs      = 8        # Number of epochs, i.e. times that we run through the entire dataset

device = pt.device('cuda' if pt.cuda.is_available() else 'cpu')
model = TestModel() 
sparse_config = {
    "seq.0": SemiSparseLinear,
}
swap_linear_with_semi_sparse_linear(model, sparse_config)
model.to(device)       # Mount the model to the selected device. Either CPU or GPU.
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_function = CrossEntropyLoss()
train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True, collate_fn=sparse_collate)
validation_loader = DataLoader(dataset=val_data, batch_size=batch_size, collate_fn=sparse_collate)

training_loss, validation_loss = Train(model, optimizer, loss_function, train_loader, validation_loader, device, n_epochs)

KeyboardInterrupt: 